# M23 — three-stream equal-budget Pareto control (train-only)

This notebook repeats the locked M5 RanPAC control on three independent streams (seeds 2025, 2026, and 2027). The stream seed controls class order, train/validation split, random-ReLU projection, and CountSketch. It reports mean ± sample standard deviation for all six methods. No CIFAR-100 test feature is materialized or read.

Select a Colab GPU (T4 recommended), upload the original `srq_generalization_m5_equal_budget_train_only.zip` when requested, and run the cells from top to bottom. The runner is resume-safe: completed stream JSON files are reused after an interruption.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='20e32ea'
WORK_DIR='/content/SOHO-CL'
RUN_ROOT='/content/srq_m23'
FEATURE_CACHE_DIR=RUN_ROOT+'/cifar_train_features'
OUTPUT_DIR=RUN_ROOT+'/output'
CONFIG='configs/srq_generalization_m23_equal_budget_multistream_train_only.json'
RUNNER='tools/srq_generalization_m23.py'
M5_NAME='srq_generalization_m5_equal_budget_train_only.zip'
M5_SHA='0f5e23fa4a7d83926638641025fb103895561073cd1f0fa4e1e0d552fd2fa931'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
FINAL_EXPORT='/content/srq_generalization_m23_equal_budget_multistream_train_only.zip'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh pinned checkout, dependency installation, GPU check, and source locks.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
def sha_raw(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
os.chdir('/content')
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git','clone','--no-checkout','--quiet',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach','--quiet',REPO_COMMIT],cwd=WORK_DIR,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK_DIR,text=True).strip().startswith(REPO_COMMIT)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a Colab GPU and restart from the first cell.'
print('GPU:',torch.cuda.get_device_name(0))
EXPECTED_SOURCE={
 'configs/srq_generalization_m23_equal_budget_multistream_train_only.json':'18d7c851fe4d867ad538ce0acdd9311809df61ce8f0282212f8e8f2badc8baf1',
 'tools/srq_generalization_m23.py':'f276ac72dd8519d82660cb8a01d98c4aa235ba33e1c663091c986252abc55755',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'methods/frontends/ranpac.py':'6b94532f607d245c0d148d09c1a36b44dbbf964f4ee9cbf05ba6f64826a90e60',
 'methods/frontends/countsketch.py':'1a35ec290ac3f7df89d0069790705456791d71ec7f17e3042103aa774af9e61d',
 'methods/frontends/__init__.py':'80cbc117f6112d278944fa05949eb982c3928142796fdcde9a1fd9b05615b7d9'
}
for path,expected in EXPECTED_SOURCE.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Pinned checkout must be clean.'
print('PINNED SOURCES: PASS')

In [ ]:
# Fast correctness gates; no CIFAR feature or test data is read.
command=[sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m23.py','tests/test_srq_generalization_m5.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M23 local gates failed; preserve the complete traceback.'
print('M23 PREFLIGHT TESTS: PASS')

In [ ]:
# Upload and verify the original one-stream M5 artifact used for the s2025 closure gate.
from google.colab import files
uploaded=files.upload()
assert M5_NAME in uploaded,'Upload the exact M5 ZIP with its original filename.'
matches=sorted(path for path in Path('/content').rglob(M5_NAME) if path.is_file())
assert len(matches)==1,f'Expected exactly one uploaded M5 artifact; found {matches}'
M5_PATH=Path('/content')/M5_NAME
if matches[0].resolve()!=M5_PATH.resolve(): shutil.move(str(matches[0]),str(M5_PATH))
assert not (Path(WORK_DIR)/M5_NAME).exists(),'Keep uploaded artifacts outside the pinned git checkout.'
assert sha_raw(M5_PATH)==M5_SHA,(sha_raw(M5_PATH),M5_SHA)
print('M5 ARTIFACT VERIFIED:',M5_PATH)

In [ ]:
# Download the locked backbone and CIFAR source, then materialize train features only.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE
assert sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m23','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN-ONLY CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run all three streams. Rerunning this cell reuses completed stream JSON files.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--original-m5-artifact',str(M5_PATH),'--require-clean-git']
print('M23 START: s2025/s2026/s2027; resume-safe per-stream outputs.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'m23_results.json'
assert result_path.is_file(),'M23 failed before writing the aggregate; preserve the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('BUDGET LOCK:',json.dumps(result['budget_lock'],indent=2))
print('AGGREGATE:',json.dumps(result['aggregate'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='PASS_M23_EQUAL_BUDGET_MULTISTREAM_TRAIN_ONLY','M23 failed; do not relax gates or inspect test accuracy.'

In [ ]:
# Export only reproducibility evidence; feature tensors stay outside the archive.
bundle=Path('/content/srq_generalization_m23_equal_budget_multistream_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(Path(OUTPUT_DIR)/'m23_results.json',bundle/'m23_results.json')
for stream_path in sorted((Path(OUTPUT_DIR)/'m23_results').glob('stream_*_results.json')): shutil.copy2(stream_path,bundle/stream_path.name)
shutil.copy2(Path(CONFIG),bundle/'config.json')
archive=shutil.make_archive(FINAL_EXPORT[:-4],'zip',root_dir=bundle)
print('ARTIFACT:',archive,'SHA256:',sha_raw(archive),'SIZE:',Path(archive).stat().st_size)
files.download(archive)